# SFJ / Media Skills – Vollprüfung und quellenbasierte Ergänzung

**Stand: 6. November 2026 · Projektbibel EVT247**

Dieses Notebook ist für **Google Colab** vorbereitet. Es verarbeitet den *vollständigen aktuellen Website-Snapshot (zuletzt 1’352 Angebote)* und die bereits vorhandene 140er-B2-Evidenz. Es liest vor jeder Ausführung den **aktuellen unveröffentlichten PR10-Stand** und die **Drive-Projektbibel**. Es verändert **weder Master noch Website**. Ein neuerer, davon abweichender Drive-Master muss vor einer späteren Datenübernahme eigens reconciliert werden.

### Bedienung

1. Zunächst den Modus **AUDIT** unverändert lassen und oben **Laufzeit → Alle ausführen** wählen. Die Google-Drive-Freigabe bestätigen. Ergebnis: vollständige Feldprüfung, URL-Liste und Drive-Backup, **0 Firecrawl-Credits**.
2. Um alle nötigen offiziellen Seiten kostenlos direkt abzurufen: Modus auf **KOSTENLOS** ändern und erneut **Alle ausführen**. Bei Verbindungsabbrüchen einfach erneut ausführen – bereits gesicherte URLs bleiben erledigt.
3. Für schwer abrufbare Quellen: Im Firecrawl-Dashboard einen **separaten API-Key mit maximal 300 Credits Ausgabenlimit** einrichten, dann Modus **MIT_FIRECRAWL** wählen, **Alle ausführen**. Der API-Key wird in einem **verdeckten Eingabefeld** abgefragt und **nicht gespeichert**. Schon erfolgreiche kostenlose URLs werden nicht erneut mit Firecrawl abgerufen.

**Sicherheit:** Keine automatischen Publikationen oder ungeprüften Änderungen an Preis, Tarif, Kursidentität und Zulassung. Veraltete Durchführungen bleiben Archivdaten. Fehlende einzelne Plätze oder die exakte Raumnummer sind allein keine generellen Freigabesperren. Das Projekt beschränkt sich auf CH/DE/AT/LI/USA.

**Ergebnis:** Ein Drive-Ordner `SFJ_COLAB_GESAMTLAUF_<Git-SHA>` mit Audit, per-URL-Belegen, Restfällen, Zahlen und SHA-verifizierten Zwischenständen. Gib den Ordnerlink nach dem Lauf hier im Chat an, damit ich die Feldänderungen nachprüfen und im richtigen Drive-Successor übernehmen kann.

**Wichtig:** Das Notebook ist von ChatGPT erstellt und lokal offline getestet. Eine Live-Ausführung in deinem Colab-/Drive-/Firecrawl-Konto ist noch nicht erfolgt.


In [ ]:
#@title 1. Einstellungen (kein Programmieren erforderlich)
MODUS = "AUDIT"  #@param ["AUDIT", "KOSTENLOS", "MIT_FIRECRAWL"]
FIRECRAWL_MAX_CREDITS = 300  #@param {type:"integer"}
MAX_FREE_URLS = 3000  #@param {type:"integer"}
MAX_FREE_WORKERS = 8  #@param {type:"integer"}
CHECKPOINT_BATCH = 30  #@param {type:"integer"}
assert MODUS in ("AUDIT","KOSTENLOS","MIT_FIRECRAWL")
assert 0 < FIRECRAWL_MAX_CREDITS <= 300
assert 1 <= MAX_FREE_URLS <= 5000 and 1 <= MAX_FREE_WORKERS <= 12
assert 1 <= CHECKPOINT_BATCH <= 100
print("Gewählter Modus:",MODUS)


In [ ]:
#@title 2. Benötigte Python-Bibliotheken installieren
%pip -q install requests beautifulsoup4 pandas pymupdf google-api-python-client google-auth-httplib2


In [ ]:
"""SFJ Media Skills: importable, offline-testable helpers for the Colab audit.
No network calls, Drive writes or Firecrawl calls on import.
"""
from __future__ import annotations
import csv
import hashlib
import io
import ipaddress
import json
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import date
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

IN_SCOPE={"schweiz","deutschland","österreich","liechtenstein","usa","vereinigte staaten", "ch","de","at","li","us"}
EMPTY_MARKERS={"", "-", "–", "n/a", "unknown", "nicht bekannt", "nicht publiziert", "nicht veröffentlicht", "keine angabe", "tbd", "k.a.", "nicht_publiziert", "on_request_format", "preisanfrage", "preis auf anfrage"}
MATERIAL_GAPS={"DELIVERY_NOT_VERIFIED","STRUCTURED_PRICE_BUDGET_UNVERIFIED","SOURCE_VERSION_OR_IDENTITY_GUARD","ACCESS_TARIFF_ELIGIBILITY_GUARD","SOURCE_PRIMARY_UNRESOLVED","OWNER_SCOPE_REVIEW","TAX_SCOPE_UNSPECIFIED"}
TRACKING_PARAMS={"fbclid","gclid","mc_cid","mc_eid","igshid"}

def sha(data):
    if isinstance(data,str): data=data.encode('utf-8')
    return hashlib.sha256(data).hexdigest()

def parse_site_js(text):
    start=text.find('[');end=text.rfind(']')
    if start<0 or end<start: raise ValueError('data.js: Angebotsliste nicht erkennbar')
    obj=json.loads(text[start:end+1]);
    if not isinstance(obj,list):raise ValueError('data.js: Liste erwartet')
    ids=[str(x.get('id','')).strip() for x in obj]
    if not all(ids) or len(set(ids))!=len(ids): raise ValueError('Leere oder doppelte Stable-Offer-ID')
    return obj

def parse_sidecar_js(text):
    start=text.find('{');end=text.rfind('}')
    if start<0 or end<start:raise ValueError('Sidecar: JSON fehlt')
    obj=json.loads(text[start:end+1]);
    if not isinstance(obj.get('byId'),dict):raise ValueError('Sidecar byId fehlt')
    return obj

def parse_tsv(text):
    rows=list(csv.DictReader(io.StringIO(text),delimiter='\t'))
    if any(None in r for r in rows):raise ValueError('TSV-Spaltenfehler')
    return rows

def normalize_url(raw):
    """Dedupe only tracking params; preserve business params, cases, route aliases."""
    try:
        s=urlsplit((raw or '').strip())
        if s.scheme.lower()!='https' or not s.hostname:return ''
        h=s.hostname.lower().rstrip('.')
        try:
            ip=ipaddress.ip_address(h)
            if not ip.is_global:return ''
        except ValueError:
            if h=='localhost' or h.endswith('.local') or '.' not in h:return ''
        if s.username or s.password:return ''
        netloc=h+((':'+str(s.port)) if s.port and s.port!=443 else '')
        q=sorted([(k,v) for k,v in parse_qsl(s.query,keep_blank_values=True)
                  if k.lower() not in TRACKING_PARAMS and not k.lower().startswith('utm_')])
        path=s.path or '/'
        return urlunsplit(('https',netloc,path,urlencode(q,doseq=True),''))
    except (ValueError,TypeError):return ''

def missing(v):
    if v is None:return True
    if isinstance(v,str):return v.strip().lower() in EMPTY_MARKERS
    return False

def clean_norm(s):
    s=unicodedata.normalize('NFKD',str(s or ''))
    s=''.join(x for x in s if not unicodedata.combining(x))
    return re.sub(r'[^a-z0-9]+',' ',s.lower()).strip()

def identity_match(title,content):
    """Conservative signal, not proof of exact edition or price."""
    a={x for x in clean_norm(title).split() if len(x)>=4 and x not in {'seminar','kurs','training','journalismus','formation','weiterbildung'}}
    b=set(clean_norm((content or '')[:35000]).split())
    if len(a)<2:return 'UNVERIFIED_SHORT_TITLE'
    fraction=len(a&b)/len(a)
    return 'TITLE_WORDS_PRESENT' if fraction>=0.72 else 'TITLE_MATCH_UNPROVEN'

def date_future(x,today=None):
    today=today or date.today()
    try:return date.fromisoformat(str(x)[:10]) >= today
    except (ValueError,TypeError):return False

def audit_offers(offers,sidecar,b2_rows=None,today=None,closed_ids=None):
    """One row per site ID; never mark whole offer as verified from prose."""
    today=today or date.today()
    byid=sidecar['byId']; expected=set(x['id'] for x in offers)
    if expected!=set(byid):
        raise ValueError(f'Sidecar-Abgleich FEHLER: Website-only={len(expected-set(byid))}, Sidecar-only={len(set(byid)-expected)}')
    b2={r['offer_id']:r for r in (b2_rows or [])}
    qa09_closed=set(closed_ids or [])
    results=[]
    for o in offers:
        id=o['id'];side=byid[id];proof=b2.get(id,{})
        core=[]
        if missing(o.get('title')):core.append('TITLE_MISSING')
        if missing(o.get('provider')):core.append('PROVIDER_MISSING')
        url=normalize_url(o.get('url',''))
        if not url:core.append('HTTPS_SOURCE_MISSING_OR_INVALID')
        if missing(o.get('format')):core.append('FORMAT_NOT_PUBLISHED')
        if missing(o.get('duration')):core.append('DURATION_NOT_PUBLISHED')
        if missing(o.get('priceDisplay')):core.append('PRICE_TEXT_NOT_PUBLISHED')
        if missing(o.get('language')):core.append('LANGUAGE_NOT_PUBLISHED')
        if missing(o.get('access')):core.append('ACCESS_TEXT_NOT_PUBLISHED')
        if missing(o.get('description')):core.append('DESCRIPTION_NOT_PUBLISHED')
        po=side.get('priceOptions') or []
        if not side.get('priceEligible',False) or not po or all(p.get('amountType') in ('NOT_PUBLISHED','ON_REQUEST',None) for p in po):
            core.append('STRUCTURED_PRICE_NOT_VERIFIED')
        delivery=side.get('delivery') or {}
        if delivery.get('evidenceTier')!='VERIFIED':core.append('DELIVERY_NOT_VERIFIED')
        tags=[v for v in str(proof.get('field_gap_tags','')).split('|') if v]
        critical=sorted(set(core+list(set(tags)&MATERIAL_GAPS)))
        sessions=o.get('sessions',None)
        future_count=sum(1 for sess in sessions if date_future(sess.get('startDate'),today)) if isinstance(sessions,list) else 0
        is_history=(isinstance(sessions,list) and len(sessions)>0 and future_count==0)
        # An ended individual course is archival, not a mandate to relaunch discovery.
        # A verified undated self-study channel stays relevant even if its old classroom sessions ended.
        verified_selfstudy=delivery.get('evidenceTier')=='VERIFIED' and any(
            v.get('temporalMode')=='ASYNCHRONOUS' and 'SELF_STUDY' in (v.get('instructionFormats') or [])
            for v in (delivery.get('variants') or []))
        if is_history and verified_selfstudy:is_history=False
        if not isinstance(sessions,list):
            end=o.get('endDate') or o.get('startDate')
            if end and not date_future(end,today):is_history=True
        allowed_country=clean_norm(o.get('country')) in {clean_norm(x) for x in IN_SCOPE}
        interesting=bool(critical) and bool(url) and not is_history and allowed_country and id not in qa09_closed
        weight=(80 if 'SOURCE_VERSION_OR_IDENTITY_GUARD' in critical else 0) + (75 if 'OWNER_SCOPE_REVIEW' in critical else 0)
        weight+=(40 if 'DELIVERY_NOT_VERIFIED' in critical else 0)+(35 if 'STRUCTURED_PRICE_NOT_VERIFIED' in critical or 'STRUCTURED_PRICE_BUDGET_UNVERIFIED' in critical else 0)
        weight+=(20 if 'ACCESS_TARIFF_ELIGIBILITY_GUARD' in critical else 0)
        weight+=min(24,len(critical)*4)
        if proof.get('priority','').startswith('P0'):weight+=25
        results.append(dict(offer_id=id,provider=o.get('provider',''),title=o.get('title',''),country=o.get('country',''),source_url=url,
                            existing_start=o.get('start',''),session_count=len(sessions) if isinstance(sessions,list) else '',future_session_count=future_count,
                            currentness='ARCHIVE_NO_FUTURE_SESSION' if is_history else 'ACTIVE_OR_UNDATED',
                            price_eligible=bool(side.get('priceEligible')),delivery_tier=delivery.get('evidenceTier','UNKNOWN'),
                            source_hold_tags='|'.join(tags),open_material_fields='|'.join(critical),
                            priority_score=weight,requires_source_research=interesting,
                            previous_publication_status=o.get('publicationStatus',''),existing_source_key=sha(url) if url else '',
                            decision='QA09_CLOSED_REUSE_ONLY' if id in qa09_closed else ('SOURCE_RESEARCH_CANDIDATE' if interesting else ('ARCHIVE_NO_AUTO_RESCAN' if is_history else 'NO_PAID_FETCH_NEEDED'))))
    return results

def source_tasks(audit):
    grouped=defaultdict(lambda:{'ids':[],'providers':set(),'score':0,'fields':set()})
    for a in audit:
        if not a['requires_source_research'] or not a['source_url']:continue
        g=grouped[a['source_url']]
        g['ids'].append(a['offer_id']);g['providers'].add(a['provider']);g['score']=max(g['score'],a['priority_score'])
        g['fields'].update(a['open_material_fields'].split('|'))
    tasks=[]
    for url,g in grouped.items():
        tasks.append(dict(url=url,ids=sorted(g['ids']),providers=sorted(g['providers']),priority=g['score']+min(len(g['ids']),12)*3,
                          missing_fields=sorted(g['fields']-set([''])),domain=urlsplit(url).hostname))
    # interleave domains to avoid 300 URLs from one provider / host, preserve priority within host
    hosts=defaultdict(list)
    for t in tasks:hosts[t['domain']].append(t)
    for z in hosts.values():z.sort(key=lambda x:(-x['priority'],x['url']))
    out=[]
    while any(hosts.values()):
        for host in sorted(hosts,key=lambda h:(-(hosts[h][0]['priority'] if hosts[h] else -1),h)):
            if hosts[host]:out.append(hosts[host].pop(0))
    return out

KEYWORD_RE={
  'PRICE':re.compile(r'(?i)(?<!\w)(?:CHF|EUR|USD|€|\$)\s*[0-9][0-9\s\x27.,]*'),
  'DATES':re.compile(r'(?i)\b\d{1,2}[./]\d{1,2}[./]\d{2,4}\b|\b20\d{2}-\d{2}-\d{2}\b'),
  'DURATION':re.compile(r'(?i)\b\d{1,3}\s*(?:tage|tagen|days?|jours?|stunden|hours?|heures?|wochen|weeks?|semesters?)\b'),
  'DELIVERY':re.compile(r'(?i)\b(?:online.?live|online|hybrid|pr[aä]senz|pr[eé]sentiel|distanciel|in.person|webinar|self.paced|selbststudium)\b'),
  'ACCESS':re.compile(r'(?i)\b(?:zulassung|voraussetzungen|admission|eligibility|inscription|anmeldung|bewerbung)\b')
}

def find_evidence(title,full_text,url):
    """Find source snippets only; no source-derived value becomes trusted by regex."""
    status=identity_match(title,full_text)
    lines=[re.sub(r'\s+',' ',s).strip() for s in (full_text or '').splitlines()]
    snippets=[]
    for field,rx in KEYWORD_RE.items():
        matching=[]
        for line in lines:
            if len(line)<5 or len(line)>800:continue
            if rx.search(line):matching.append(line[:420])
        for line in matching[:5]:
            snippets.append(dict(field=field,source_url=url,excerpt=line,identity_signal=status,
                                 decision='EVIDENCE_ONLY_REQUIRES_FIELD_AND_EDITION_ADJUDICATION'))
    return snippets


def compact_stats(audit,tasks):
    c=Counter(a['decision'] for a in audit)
    p=Counter(x for a in audit for x in a['open_material_fields'].split('|') if x)
    return {'offers':len(audit),'unique_urls_research_candidates':len(tasks),'by_decision':dict(c),'most_common_gaps':p.most_common(18),
            'delivery_not_verified':sum(a['delivery_tier']!='VERIFIED' for a in audit),
            'structured_price_eligible':sum(a['price_eligible'] for a in audit),
            'historical_offers':sum(a['currentness']=='ARCHIVE_NO_FUTURE_SESSION' for a in audit)}


def csv_bytes(rows,fields=None):
    fields=fields or (list(rows[0]) if rows else ['empty'])
    io_obj=io.StringIO(newline='');w=csv.DictWriter(io_obj,fieldnames=fields,delimiter='\t',extrasaction='ignore')
    w.writeheader();
    for row in rows:w.writerow({k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v for k,v in row.items()})
    return io_obj.getvalue().encode('utf-8')


In [ ]:
"""Colab runtime for SFJ. Requires helpers in sfj_colab_core.py cell above.
Never auto-modifies master data, GitHub, or hosted website.
"""
import io, os, json, zipfile, hashlib, time, re, threading, getpass, socket
from collections import Counter
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlsplit
from urllib.robotparser import RobotFileParser
import requests
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload
from IPython.display import display, Markdown

REPO='george-sarpong/weiterbildungsradar-journalismus'
PR_NO=10
BIBLE_ID='1zOTSwhCC8-1IQDxI5HgwpPC__LAc2zGuDXFtM2Pj-Cs'
KNOWLEDGE_ID='1qW3FcD6Vd90zvbI_kVDIw49y1-mShi6ZOHG3G1i4o6Q'
QA09_ROSTER_ID='1kvL0WkdffjOB0BytsP66nuwRpEbjgmfZQPAblowkZGY'
USER_AGENT='SFJ-MediaSkills-Research/1.0 (https://mediaskills.ch; official-source-validation)'
PAID_URL='https://api.firecrawl.dev/v2/scrape'
CREDIT_URL='https://api.firecrawl.dev/v2/team/credit-usage'
print('Google Drive wird einmalig autorisiert. Es werden ausschliesslich neue Ergebnisdateien in einem eigenen Arbeitsordner erzeugt.')
auth.authenticate_user()
credentials, _=google.auth.default()
drive_api=build('drive','v3',credentials=credentials,cache_discovery=False)
docs_api=build('docs','v1',credentials=credentials,cache_discovery=False)


def get_http_json(url,headers=None):
    r=requests.get(url,timeout=25,headers=headers or {'Accept':'application/vnd.github+json'})
    r.raise_for_status();return r.json()

p=get_http_json(f'https://api.github.com/repos/{REPO}/pulls/{PR_NO}')
if p.get('state')!='open' or not p.get('draft') or p.get('merged'):
    raise RuntimeError('PR10 ist nicht mehr OPEN/DRAFT/UNMERGED; ChatGPT muss erst den Successor pruefen.')
HEAD=p['head']['sha']
print('Aktueller unveroeffentlichter GitHub PR10-Successor:',HEAD)

doc=docs_api.documents().get(documentId=BIBLE_ID,includeTabsContent=True).execute()
def iter_doc_text(obj):
    if isinstance(obj,dict):
        if 'textRun' in obj and isinstance(obj['textRun'],dict):yield obj['textRun'].get('content','')
        for k,v in obj.items():
            if k!='textRun':yield from iter_doc_text(v)
    elif isinstance(obj,list):
        for v in obj:yield from iter_doc_text(v)
bible_text=''.join(iter_doc_text(doc))
if 'EVT-BIBEL-20261106-247' not in bible_text:
    raise RuntimeError('Aktuelle Drive-Projektbibel EVT247 nicht bestaetigt; kein Crawl starten.')
print('Drive-Bibel gelesen, letzter vereinbarter Methodenbeschluss EVT247 vorhanden. Bibel-Revision:',doc.get('revisionId','?'))
qa_doc=docs_api.documents().get(documentId=QA09_ROSTER_ID,includeTabsContent=True).execute()
qa09_text=''.join(iter_doc_text(qa_doc))
qa09_closed={m.group(2).strip() for m in re.finditer(r'(?:^|\n)\s*(\d{3})\s*\|\s*([^|\n]+?)\s*\|',qa09_text)}
if len(qa09_closed)!=116:
    raise RuntimeError(f'Der QA09-Freeze-Roster ist nicht eindeutig 116: {len(qa09_closed)}; Wiederrecherche unterbunden.')
print('QA09-Freeze aus Drive gelesen: 116/116 IDs werden NICHT neu recherchiert.')

def github_raw(path):
    url=f'https://raw.githubusercontent.com/{REPO}/{HEAD}/{path}'
    r=requests.get(url,timeout=60,headers={'User-Agent':USER_AGENT})
    r.raise_for_status();return r.text

raw_offers=github_raw('data.js')
raw_sidecar=github_raw('smart-search-data.js')
raw_b2=github_raw('docs/b2/all-140-website-gold-field-gates-20261105.tsv')
offers=parse_site_js(raw_offers)
sidecar=parse_sidecar_js(raw_sidecar)
b2_rows=parse_tsv(raw_b2)
if len(offers)!=1352 or len(b2_rows)!=140:
    print(f'ACHTUNG: Der Live-Successor hat andere Zahlen als die historische Baseline: Website={len(offers)}; B2={len(b2_rows)}. Die aktuell gelesenen Daten werden vollstaendig geprueft.')
if len(set(x['offer_id'] for x in b2_rows))!=len(b2_rows):
    raise RuntimeError('B2-IDs nicht eindeutig')
if not set(x['offer_id'] for x in b2_rows).issubset(set(x['id'] for x in offers)):
    raise RuntimeError('B2-Datei passt nicht zur aktuellen Website-Offerliste; bitte vor der Recherche einen aktuellen Crosswalk in Drive klaeren')

audit_rows=audit_offers(offers,sidecar,b2_rows,today=datetime.now(ZoneInfo('Europe/Zurich')).date(),closed_ids=qa09_closed)
tasks=source_tasks(audit_rows)
report=compact_stats(audit_rows,tasks)
report.update({'source_git_sha':HEAD,'source_data_sha256':sha(raw_offers),'source_sidecar_sha256':sha(raw_sidecar),
               'source_b2_sha256':sha(raw_b2),'bible_revision':doc.get('revisionId'),
               'scope':'CH DE AT LI USA','publication':'NO_MASTER_MAIN_OR_SITE_WRITES',
               'firecrawl_balance_verified':False,'qa09_frozen_roster':len(qa09_closed),
               'qa09_research_protected':sum(x['offer_id'] in qa09_closed for x in audit_rows)})
print('\nVOLLPRUEFUNG 0 Credits – aktueller Website-Arbeitsbestand, Drive-Bibel gelesen (kein universeller Drive-Master-Merge):')
print(json.dumps(report,ensure_ascii=False,indent=2))
display(pd.DataFrame([{'Angebote im Website-Snapshot':len(audit_rows),'B2-Teilmenge':len(b2_rows),
                       'Relevante eindeutige URLs':len(tasks),'Davon historisch':report['historical_offers']}]))


def folder_by_name(name):
    safe=name.replace("'","\\'")
    q=f"name = '{safe}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    fs=drive_api.files().list(q=q,fields='nextPageToken,files(id,name)',page_size=100).execute().get('files',[])
    if len(fs)>1:raise RuntimeError('Mehrere gleichnamige Arbeitsordner: bitte in Drive eindeutige Ordner schaffen.')
    if fs:return fs[0]['id']
    created=drive_api.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder'},fields='id').execute()
    return created['id']

FOLDER=f'SFJ_COLAB_GESAMTLAUF_{HEAD[:12]}'
folder_id=folder_by_name(FOLDER)
print('Drive-Arbeitsordner:',f'https://drive.google.com/drive/folders/{folder_id}')


def find_file(name):
    name_escaped=name.replace("'","\\'")
    q=f"name = '{name_escaped}' and '{folder_id}' in parents and trashed = false"
    files=[];token=None
    while True:
        resp=drive_api.files().list(q=q,fields='nextPageToken,files(id,name,md5Checksum)',page_size=100,page_token=token).execute()
        files.extend(resp.get('files',[]));token=resp.get('nextPageToken')
        if not token:break
    if len(files)>1:raise RuntimeError('Doppelte Dateinamen im Arbeitsordner: '+name)
    return files[0] if files else None

def remote_bytes(file_id):
    buffer=io.BytesIO();media=drive_api.files().get_media(fileId=file_id)
    downloader=MediaIoBaseDownload(buffer,media,chunksize=1024*1024)
    done=False
    while not done:_,done=downloader.next_chunk(num_retries=2)
    return buffer.getvalue()

def write_immutable(name,data):
    if isinstance(data,str):data=data.encode('utf-8')
    expected=sha(data);found=find_file(name)
    if found:
        if sha(remote_bytes(found['id']))!=expected:raise RuntimeError('Persistenz-Konflikt bei '+name)
        return found['id']
    payload=MediaIoBaseUpload(io.BytesIO(data),mimetype='application/zip',resumable=False)
    res=drive_api.files().create(body={'name':name,'parents':[folder_id]},media_body=payload,fields='id').execute()
    file_id=res['id']
    if sha(remote_bytes(file_id))!=expected:raise RuntimeError('Drive Readback SHA-256 MISMATCH: '+name)
    return file_id

def make_zip(files):
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
        for name,content in files.items():
            if isinstance(content,str):content=content.encode('utf-8')
            z.writestr(name,content)
    return buf.getvalue()

def jsonl_bytes(rows):return ('\n'.join(json.dumps(x,ensure_ascii=False) for x in rows)+'\n').encode('utf-8')

snapshot=make_zip({'angebote_audit.tsv':csv_bytes(audit_rows),'einmalige_quellenauftraege.tsv':csv_bytes(tasks),
                   'summary.json':json.dumps(report,ensure_ascii=False,indent=2),
                   'methoden.txt':'Website-Kandidatenstand, nicht automatisch der aktuelle Drive-Master; Quelle PR10 Draft. Keine Master-, Main- oder Publikationsschreiboperation.\n'})
audit_file=f'AUDIT_{HEAD[:12]}_{sha(snapshot)[:12]}.zip'
write_immutable(audit_file,snapshot)
print('Audit samt 1.352 Einzelzeilen in Drive gesichert und SHA-256-zurueckgelesen:',audit_file)

if MODUS=='AUDIT':
    print('Modus AUDIT: 0 Firecrawl-Credits, keine Webseitenaufrufe ausser GitHub und Google Drive.')
else:
    # USER TOOL: This phase is only entered after a consciously selected research mode.
    pass


In [ ]:
# This cell is deliberately inert in AUDIT mode.
if MODUS != 'AUDIT':
    print('\nRECHERCHE: Bereits abgeschlossene URLs werden nicht erneut aufgerufen. Keine Master-/Website-Schreibzugriffe.')

    def list_work_files():
        q=f"'{folder_id}' in parents and trashed = false"
        found=[];token=None
        while True:
            z=drive_api.files().list(q=q,fields='nextPageToken,files(id,name)',page_size=1000,page_token=token).execute()
            found.extend(z.get('files',[]));token=z.get('nextPageToken')
            if not token:break
        return found

    def zip_jsonl(blob):
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            if 'results.jsonl' not in z.namelist():return []
            return [json.loads(s) for s in z.read('results.jsonl').decode('utf-8').splitlines() if s.strip()]

    # Read back previous results (including from earlier disconnected Colab runtimes).
    files=list_work_files()
    free_results={};paid_results={};reservations={}
    for f in files:
        n=f['name']
        if n.startswith(('FREE_'+HEAD[:12]+'_','PAID_'+HEAD[:12]+'_')) and n.endswith('.zip'):
            for record in zip_jsonl(remote_bytes(f['id'])):
                if n.startswith('FREE_'): free_results[record['url']]=record
                else: paid_results[record['url']]=record
        elif n.startswith('RESERVE_'+HEAD[:12]+'_') and n.endswith('.zip'):
            for rec in zip_jsonl(remote_bytes(f['id'])):
                reservations[rec['url']]=rec
    print(f'Wiederaufnahme: {len(free_results)} kostenlose URLs, {len(paid_results)} bezahlte Ergebnisse, {len(reservations)} reservierte Firecrawl-Anfragen.')

    # Provider-friendly direct fetching: no client-side browser bypass; no retry loops.
    ROBOT_LOCK=threading.Lock();ROBOTS={}
    def robot_allowed(url):
        origin=f'{urlsplit(url).scheme}://{urlsplit(url).netloc}'
        with ROBOT_LOCK:
            if origin in ROBOTS:return ROBOTS[origin].can_fetch(USER_AGENT,url) if ROBOTS[origin] else True
            try:
                rr=requests.get(origin+'/robots.txt',timeout=(6,12),headers={'User-Agent':USER_AGENT})
                if rr.status_code==200:
                    rp=RobotFileParser();rp.parse(rr.text.splitlines());ROBOTS[origin]=rp
                    return rp.can_fetch(USER_AGENT,url)
                if rr.status_code in (401,403):
                    rp=RobotFileParser();rp.disallow_all=True;ROBOTS[origin]=rp;return False
                ROBOTS[origin]=None;return True
            except requests.RequestException:
                ROBOTS[origin]=None;return False  # uncertain robots -> do not try a paid bypass

    HOST_LOCK=threading.Lock();HOST_NEXT={}
    def host_pace(url):
        host=urlsplit(url).hostname
        with HOST_LOCK:
            delay=max(0,HOST_NEXT.get(host,0)-time.monotonic())
            HOST_NEXT[host]=max(time.monotonic(),HOST_NEXT.get(host,0))+1.0
        if delay:time.sleep(delay)

    def free_fetch(t):
        url=t['url'];timestamp=datetime.now(timezone.utc).isoformat()
        result={'url':url,'method':'FREE_DIRECT','status':'UNKNOWN','fetched_at':timestamp,'body':'','body_sha256':'','metadata_status':'',
                'offer_ids':t['ids'],'note':''}
        if not robot_allowed(url):
            result.update(status='ROBOTS_NOT_ALLOWED_OR_UNKNOWN',note='robots.txt disallow or not verifiable; no paid bypass');return result
        host_pace(url)
        try:
            with requests.get(url,timeout=(9,24),stream=True,allow_redirects=True,
                              headers={'User-Agent':USER_AGENT,'Accept':'text/html,application/pdf,text/plain,*/*'}) as resp:
                result['metadata_status']=resp.status_code
                if not normalize_url(resp.url):
                    result.update(status='REDIRECT_NOT_HTTPS');return result
                if resp.status_code!=200:
                    result.update(status='HTTP_'+str(resp.status_code));return result
                ctype=resp.headers.get('Content-Type','').lower()
                chunks=[];total=0
                for chunk in resp.iter_content(chunk_size=65536):
                    total+=len(chunk)
                    if total>6_000_000:
                        result.update(status='CONTENT_TOO_LARGE');return result
                    chunks.append(chunk)
                content=b''.join(chunks)
                if ctype.startswith('application/pdf') or content[:4]==b'%PDF':
                    import fitz
                    pdf=fitz.open(stream=content,filetype='pdf')
                    text='\n'.join(p.get_text() for p in pdf[:min(len(pdf),60)])
                    result['method']='FREE_PDF'
                elif 'html' in ctype or b'<html' in content[:3000].lower():
                    soup=BeautifulSoup(content,'html.parser')
                    for bad in soup(['script','style','noscript','nav','footer']):bad.decompose()
                    main=soup.select_one('main, article, [role=main]') or soup.body or soup
                    text=main.get_text('\n',strip=True)
                else:
                    text=content.decode('utf-8',errors='replace')
                text=text[:85000]
                result.update(body=text,body_sha256=sha(text),status='CONTENT_OK' if len(text)>=220 else 'CONTENT_INSUFFICIENT')
                if not text.strip():result['status']='EMPTY_TEXT'
        except (Exception,) as e:
            # Sensitive keys never appear in this function.
            result.update(status='FETCH_FAILED',note=type(e).__name__+': '+str(e)[:140])
        return result

    remaining=[t for t in tasks if t['url'] not in free_results]
    if len(remaining)>MAX_FREE_URLS:
        print(f'Achtung: kostenloser Seitenabruf auf {MAX_FREE_URLS} von {len(remaining)} ausstehenden URLs beschraenkt.')
        remaining=remaining[:MAX_FREE_URLS]
    print('Starte kostenlose offizielle Direktabrufe fuer',len(remaining),'eindeutige URLs, in Host-schonender Parallelverarbeitung.')
    for base in range(0,len(remaining),CHECKPOINT_BATCH):
        batch=remaining[base:base+CHECKPOINT_BATCH]
        new=[]
        with ThreadPoolExecutor(max_workers=MAX_FREE_WORKERS) as ex:
            for future in as_completed([ex.submit(free_fetch,t) for t in batch]):new.append(future.result())
        # Immutable batch checkpoint BEFORE further work; deterministic name from exact URL set.
        suffix=sha('\n'.join(sorted(t['url'] for t in batch)))[:14]
        filename=f'FREE_{HEAD[:12]}_{suffix}.zip'
        write_immutable(filename,make_zip({'results.jsonl':jsonl_bytes(new)}))
        for rec in new:free_results[rec['url']]=rec
        print(f'Kostenlos erledigt: {len(free_results)}/{len(tasks)} URLs (letzter Batch gespeichert und gelesen).')

    credits_at_start=None;credits_at_end=None;key=None
    if MODUS=='MIT_FIRECRAWL':
        # No key is requested, logged or persisted in AUDIT/KOSTENLOS mode.
        if not (0<FIRECRAWL_MAX_CREDITS<=300):
            raise RuntimeError('Initialbudget muss 1–300 Credits sein. Keine bezahlten Abfragen gestartet.')
        key=getpass.getpass('Firecrawl-API-Key (verdeckt, nur im Arbeitsspeicher): ').strip()
        if not key.startswith('fc-'):
            key=None;raise RuntimeError('Kein gueltiger Firecrawl-API-Key eingegeben.')
        hdr={'Authorization':'Bearer '+key,'Content-Type':'application/json'}
        def credits():
            r=requests.get(CREDIT_URL,headers=hdr,timeout=25)
            r.raise_for_status();info=r.json()
            if not info.get('success') or 'remainingCredits' not in info.get('data',{}):
                raise RuntimeError('Credit-Saldo nicht sicher abfragbar. Ausgaben gesperrt.')
            return float(info['data']['remainingCredits'])
        credits_at_start=credits()
        print('Firecrawl-Gesamtsaldo (API-geprueft):',credits_at_start)
        print('Vor Paid-Lauf: den separaten Key im Firecrawl-Dashboard auf max. 300 Credits begrenzen; Smart Upgrade deaktivieren.')
        # Resume safety: every intended paid URL was reserved in Drive BEFORE the request.
        safe_max=FIRECRAWL_MAX_CREDITS-len(reservations)
        safe_max=max(0,min(safe_max,int(max(0,credits_at_start-30))))
        paid_targets=[t for t in tasks if t['url'] in free_results
                      and free_results[t['url']]['status'] in ('CONTENT_INSUFFICIENT','EMPTY_TEXT','HTTP_403','FETCH_FAILED')
                      and t['url'] not in reservations and t['url'] not in paid_results]
        paid_targets=paid_targets[:safe_max]
        print(f'Firecrawl-Grenze: {FIRECRAWL_MAX_CREDITS}, bereits reserviert: {len(reservations)}, neue Abfragen maximal: {len(paid_targets)}.')
        PAID_PACE_LOCK=threading.Lock();PAID_NEXT=[time.monotonic()]
        def paid_pace():
            # Conservative 10 calls/minute: safe even with Free plan quotas.
            with PAID_PACE_LOCK:
                now=time.monotonic();delay=max(0,PAID_NEXT[0]-now)
                PAID_NEXT[0]=max(now,PAID_NEXT[0])+6.1
            if delay:time.sleep(delay)
        def firecrawl_scrape(t):
            x={'url':t['url'],'method':'FIRECRAWL_MARKDOWN','status':'UNKNOWN','fetched_at':datetime.now(timezone.utc).isoformat(),
               'body':'','body_sha256':'','metadata_status':'','offer_ids':t['ids'],'note':''}
            try:
                paid_pace()
                r=requests.post(PAID_URL,headers=hdr,json={'url':t['url'],'formats':['markdown'],
                    'onlyMainContent':True,'timeout':30000},timeout=65)
                x['metadata_status']=r.status_code
                if r.status_code!=200:
                    x.update(status='API_'+str(r.status_code),note=r.text[:180]);return x
                j=r.json();obj=j.get('data') or j
                text=(obj.get('markdown') or '')[:85000]
                provider_status=(obj.get('metadata') or {}).get('statusCode')
                x['metadata_status']=provider_status or 200
                x.update(body=text,body_sha256=sha(text),status='CONTENT_OK' if len(text)>=220 and (not provider_status or provider_status==200) else 'CONTENT_INSUFFICIENT')
            except Exception as e:
                x.update(status='PAID_REQUEST_UNCERTAIN',note=type(e).__name__+': '+str(e)[:110])
            return x
        for base in range(0,len(paid_targets),2):
            batch=paid_targets[base:base+2]
            token=sha('\n'.join(sorted(t['url'] for t in batch)))[:14]
            reserve=f'RESERVE_{HEAD[:12]}_{token}.zip'
            paid=f'PAID_{HEAD[:12]}_{token}.zip'
            # Reservation has hard write+readback; missing results are NOT retried automatically after crash.
            write_immutable(reserve,make_zip({'results.jsonl':jsonl_bytes([{'url':t['url'],'ids':t['ids'],'reserved_at':datetime.now(timezone.utc).isoformat()} for t in batch])}))
            for t in batch:reservations[t['url']]=t
            with ThreadPoolExecutor(max_workers=2) as ex:
                got=list(ex.map(firecrawl_scrape,batch))
            write_immutable(paid,make_zip({'results.jsonl':jsonl_bytes(got)}))
            for x in got:paid_results[x['url']]=x
            print(f'Firecrawl reserviert/versucht: {len(reservations)}/{FIRECRAWL_MAX_CREDITS}; gespeichert: {len(paid_results)}.')
        credits_at_end=credits()
        # Drop API key reference as early as possible. Requests already completed.
        key=None;hdr=None
        print('Firecrawl-Saldo nach den Abrufen:',credits_at_end,
              '(globale Teamnutzung kann andere Schluessel einschliessen; asynchrone Buchungen koennen nachlaufen).')

    byurl={t['url']:t for t in tasks}
    best={**free_results}
    for u,p in paid_results.items():
        if p['status']=='CONTENT_OK' or u not in best or best[u]['status']!='CONTENT_OK':best[u]=p
    evidence=[];unresolved=[];brief=[]
    offer_lookup={x['id']:x for x in offers}
    for t in tasks:
        x=best.get(t['url'])
        if x is None:
            unresolved.append({'url':t['url'],'offer_ids':'|'.join(t['ids']),'reason':'NOT_ATTEMPTED_YET','required_fields':'|'.join(t['missing_fields'])})
            continue
        brief.append({'url':t['url'],'offer_ids':'|'.join(t['ids']),'status':x['status'],'method':x['method'],
                      'fetched_at':x.get('fetched_at'),'sha256':x.get('body_sha256','')})
        if x['status']!='CONTENT_OK':
            pending='PAID_RESULT_UNCERTAIN_NO_AUTO_RETRY' if t['url'] in reservations and t['url'] not in paid_results else x['status']
            unresolved.append({'url':t['url'],'offer_ids':'|'.join(t['ids']),'reason':pending,'required_fields':'|'.join(t['missing_fields'])})
            continue
        for id in t['ids']:
            o=offer_lookup[id]
            facts=find_evidence(o.get('title'),x['body'],t['url'])
            if not facts:
                unresolved.append({'url':t['url'],'offer_ids':id,'reason':'NO_EXTRACTABLE_EXACT_FIELD_EVIDENCE','required_fields':'|'.join(t['missing_fields'])})
                continue
            for fact in facts:evidence.append({'offer_id':id,**fact,'old_website_value':str(o.get({'PRICE':'priceDisplay','DURATION':'duration','DELIVERY':'format','ACCESS':'access','DATES':'start'}.get(fact['field']),'') or '')[:350]})
    for url in reservations:
        if url not in paid_results:
            unresolved.append({'url':url,'offer_ids':'|'.join(byurl.get(url,{}).get('ids',[])),
                'reason':'PAID_RESERVED_UNCERTAIN_NO_AUTO_RETRY','required_fields':'|'.join(byurl.get(url,{}).get('missing_fields',[]))})
    summary={**report,'mode':MODUS,'free_results':len(free_results),'paid_reservations':len(reservations),
             'paid_results':len(paid_results),'firecrawl_max_credits':FIRECRAWL_MAX_CREDITS,
             'credit_balance_before':credits_at_start,'credit_balance_after':credits_at_end,
             'evidence_snippet_candidates':len(evidence),'unresolved_records':len(unresolved),
             'conclusion':'EVIDENCE_CANDIDATES_ONLY_NOT_AUTOMATIC_WEBSITE_GOLD_OR_PUBLISHED',
             'created_at':datetime.now(timezone.utc).isoformat()}
    result_zip=make_zip({'gepruefte_angebote.tsv':csv_bytes(audit_rows),
                         'quellen_status.tsv':csv_bytes(brief),
                         'belegkandidaten_mit_originaltext.tsv':csv_bytes(evidence),
                         'restfaelle.tsv':csv_bytes(unresolved),
                         'kennzahlen.json':json.dumps(summary,ensure_ascii=False,indent=2)})
    result_name=f'ERGEBNIS_{HEAD[:12]}_{MODUS}_{sha(result_zip)[:12]}.zip'
    write_immutable(result_name,result_zip)
    print('\nABSCHLUSS – Ergebnisdatei in Google Drive gespeichert und SHA-256-geprueft:')
    print(f'https://drive.google.com/drive/folders/{folder_id}')
    print(json.dumps(summary,ensure_ascii=False,indent=2))
    display(pd.DataFrame([{'Angebote':len(audit_rows),'Gratisquellen':len(free_results),
                           'Firecrawl-Versuche reserviert':len(reservations),
                           'Belegkandidaten':len(evidence),'Restfall-Zeilen':len(unresolved)}]))


## Was tun nach dem Lauf?

In der Ausgabe steht ein **Google-Drive-Ordnerlink**. Dort findest du die gezippten, zurückgelesenen Ergebnisse. Du kannst mir diesen Ordnerlink schicken; dadurch muss ich nicht noch einmal dieselben Quellen abrufen.

Die Datei **`angebote_audit.tsv`** (im AUDIT-ZIP) erfasst alle Website-Angebote, inklusive historischer Ausgaben. **`belegkandidaten_mit_originaltext.tsv`** und **`restfaelle.tsv`** (im ERGEBNIS-ZIP) enthalten konkrete Quellenhinweise und wirkliche Restlücken. Ein Regex-Suchtreffer ist **kein** freigegebener Preis oder bestätigter freier Platz. Bei einem widersprüchlichen Quellbeleg wird die Kandidatenliste nicht automatisch in die Website gemergt.

Das Notebook verwendet bei Firecrawl ausschliesslich `POST /v2/scrape` mit `formats=["markdown"]` und vorherige reservierte, SHA-verifizierte Drive-Checkpoints. Nach einem Abbruch werden potenziell bereits berechnete kostenpflichtige URLs nicht automatisch wiederholt. Der eigene Firecrawl-API-Key muss zusätzlich im Dashboard begrenzt werden. Ein bestätigter einzelner API-Key kann die globale Kontonutzung anderer Schlüssel nicht kontrollieren.

Quellen: [Firecrawl Billing](https://github.com/firecrawl/firecrawl-docs/blob/main/billing.mdx) · [Firecrawl Scrape](https://github.com/firecrawl/firecrawl-docs/blob/main/advanced-scraping-guide.mdx) · [Colab FAQ](https://research.google.com/colaboratory/faq.html).
